# Predicting how often a family went without enough food

**Matthias Muscat · ICS5110 · Multinomial logistic regression**

Given six background answers, can the model distinguish **never**, **rarely**, **sometimes** and **often** going without enough food over the preceding 12 months? It learns a score for each answer and converts the four scores into probabilities. The most probable answer is the four-class prediction. The model treats the answers as distinct classes. We also measure how far a prediction is from the observed answer on the ordered scale.

**The saved run uses 600 fictional respondents.** Its purpose is to show that training, prediction and evaluation work end to end. The group is reviewing the four-category target and a common split for the actual survey. [Dataset, class counts and rebuild instructions](../work/muscat/README.md)

## Choose the data and split

The default example runs from a fresh checkout. For a group run, rebuild or download the private four-category CSV and select `survey` below. If the team adopts the proposed 80/20 split, leave `SPLIT_CSV` empty. To use another shared train/test assignment, enter the repository-relative path to its CSV. It takes two columns, `meta_source_row` and `split` (`train` or `test`). The assignment file may list respondents in any order. The notebook matches it by source-row key and checks the survey CSV against the published SHA-256 checksum.

In [1]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, confusion_matrix, f1_score, mean_absolute_error, precision_recall_fscore_support,
)
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

DATA_SOURCE = "demo"  # "demo" or "survey"
SPLIT_CSV = None     # optional path to a shared assignment file

root = next(
    (p for p in (Path.cwd(), *Path.cwd().parents) if (p / "work/muscat/q51_dataset.json").is_file()),
    None,
)
if root is None:
    raise FileNotFoundError("Run this notebook from the repository root or its 03_code directory.")
member = root / "work" / "muscat"
sys.path.insert(0, str(member))
from q51_model_data import DATASET, FEATURES, TARGET, CODES, LABELS
from q51_model_data import align_split, example_data, load_survey, proposed_split

if DATA_SOURCE == "demo":
    data = example_data()
    source_label = "Synthetic demonstration"
elif DATA_SOURCE == "survey":
    data = load_survey(member / "private" / DATASET["filename"])
    source_label = "WVS Wave 7 survey"
else:
    raise ValueError("Choose demo or survey.")

assignments = pd.read_csv(root / SPLIT_CSV) if SPLIT_CSV is not None else proposed_split(data)
partition = align_split(data, assignments)
train = partition.eq("train")
test = partition.eq("test")
X_train, X_test = data.loc[train, FEATURES], data.loc[test, FEATURES]
y_train, y_test = data.loc[train, TARGET], data.loc[test, TARGET]

counts = pd.crosstab(data[TARGET], partition).reindex(index=CODES, columns=["train", "test"], fill_value=0)
counts.index = LABELS
print(f"{source_label} | {len(data):,} rows | {len(FEATURES)} inputs")
print(counts.to_string())

Synthetic demonstration | 600 rows | 18 inputs
split      train  test
Never        285    71
Rarely        74    18
Sometimes     62    16
Often         59    15


## Fit the four-class model

A pipeline learns the mean and scale of each of the 18 inputs from the training records, then fits multinomial logistic regression with L2 regularisation (`C=1`). Fixing this setting in advance keeps the test set available for evaluation. The comparison model always predicts the most common training answer. The baseline uses the 18 inputs listed in the dataset record. Q54 is reserved for a separate experiment. The survey weights are kept for later weighting analyses.

In [2]:
model = make_pipeline(
    StandardScaler(),
    LogisticRegression(solver="lbfgs", C=1.0, max_iter=1000),
)
model.fit(X_train, y_train)
baseline = DummyClassifier(strategy="most_frequent").fit(X_train, y_train)

predicted = model.predict(X_test)
probabilities = pd.DataFrame(model.predict_proba(X_test), index=y_test.index, columns=model.classes_)[CODES]
if not np.isfinite(probabilities).all().all() or not np.allclose(probabilities.sum(axis=1), 1):
    raise ValueError("Expected four finite class probabilities that sum to one.")

print(f"{source_label} | probability columns in model order: {', '.join(LABELS)}")
if DATA_SOURCE == "demo":
    example = probabilities.iloc[:1].copy()
    example.columns = LABELS
    print("One fictional test respondent")
    print(example.round(3).to_string(index=False))

Synthetic demonstration | probability columns in model order: Never, Rarely, Sometimes, Often
One fictional test respondent
 Never  Rarely  Sometimes  Often
 0.678   0.174      0.108  0.039


## Compare with the baseline

**Macro-F1** gives each answer equal importance. **Recall** shows how many people in each observed class the model identifies. This is particularly important for “often”, the smallest survey class. **Mean absolute class error** counts the average distance between answers on the four-step scale. Predicting “never” for someone who answered “often” is three class steps away. The steps describe rank, not a count of missed meals. These initial metrics give each test respondent equal weight.

In [3]:
scores = []
for name, guesses in [("Most frequent answer", baseline.predict(X_test)), ("Logistic regression", predicted)]:
    scores.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, guesses),
        "Macro-F1": f1_score(y_test, guesses, labels=CODES, average="macro", zero_division=0),
        "Mean absolute class error": mean_absolute_error(y_test, guesses),
    })

print(f"{source_label} | held-out scores")
print(pd.DataFrame(scores).set_index("Model").round(3).to_string())

Synthetic demonstration | held-out scores
                      Accuracy  Macro-F1  Mean absolute class error
Model                                                              
Most frequent answer     0.592     0.186                      0.792
Logistic regression      0.600     0.324                      0.642


In [4]:
precision, recall, f1, support = precision_recall_fscore_support(
    y_test, predicted, labels=CODES, zero_division=0,
)
by_answer = pd.DataFrame(
    {"Precision": precision, "Recall": recall, "F1": f1, "Test respondents": support}, index=LABELS,
)
mistakes = pd.DataFrame(
    confusion_matrix(y_test, predicted, labels=CODES),
    index=pd.Index(LABELS, name="Actual"), columns=pd.Index(LABELS, name="Predicted"),
)
print(f"{source_label} | logistic regression by observed answer")
print(by_answer.round(3).to_string())
print("\nConfusion matrix (rows = actual, columns = predicted)")
print(mistakes.to_string())

Synthetic demonstration | logistic regression by observed answer
           Precision  Recall     F1  Test respondents
Never          0.710   0.930  0.805                71
Rarely         0.091   0.056  0.069                18
Sometimes      0.400   0.125  0.190                16
Often          0.273   0.200  0.231                15

Confusion matrix (rows = actual, columns = predicted)
Predicted  Never  Rarely  Sometimes  Often
Actual                                    
Never         66       4          1      0
Rarely        13       1          1      3
Sometimes      6       3          2      5
Often          8       3          1      3


## Explore misses and false alarms

For a separate alert about **recurring shortages**, add the probabilities of “sometimes” and “often”. A miss is an observed recurring shortage that falls below the alert threshold. A false alarm is a “never” or “rarely” answer at or above that threshold. This gives the dashboard a threshold to vary while keeping the original four-category prediction. The three thresholds here illustrate the trade-off. Choose an operating threshold using the training data and error costs before final test evaluation.

In [5]:
recurring_probability = probabilities[[2, 3]].sum(axis=1)
observed_recurring = y_test.isin([2, 3])
tradeoffs = []
for threshold in (0.25, 0.50, 0.75):
    flagged = recurring_probability.ge(threshold)
    tradeoffs.append({
        "Threshold": threshold,
        "Misses": int((observed_recurring & ~flagged).sum()),
        "False alarms": int((~observed_recurring & flagged).sum()),
    })
print(f"{source_label} | recurring-shortage alert")
print(pd.DataFrame(tradeoffs).to_string(index=False))

Synthetic demonstration | recurring-shortage alert
 Threshold  Misses  False alarms
      0.25       9            23
      0.50      19             6
      0.75      29             2


## Sources

- Haerpfer et al. (2022), *World Values Survey Wave 7 Cross-National Data-Set*, v6.0.0. [Dataset](https://doi.org/10.14281/18241.24)
- Scikit-learn 1.8.0. [Logistic regression](https://scikit-learn.org/1.8/modules/generated/sklearn.linear_model.LogisticRegression.html) and [stratified train/test split](https://scikit-learn.org/1.8/modules/generated/sklearn.model_selection.train_test_split.html).
- Matthias Muscat’s [decision and AI-use record](../work/muscat/log.md).